# QFF 2026 Phase 2 — Quantum Chemistry starter notebook

Shows the **official measurement workflow** with the kit's tools. It does not solve any problem statement.
Run from the `starter/` folder in the dedicated chemistry environment (`pip install -r ../requirements.txt`).

In [ ]:
import sys, json, numpy as np
from scipy.optimize import minimize
import qiskit, qiskit_aer
from qff_chem import *
print('qiskit', qiskit.__version__, '| aer', qiskit_aer.__version__)
SEED = 42

## 1. Load a locked operator and its references

In [ ]:
op, const, meta = load_operator('../data/c1/h2_0.735.json')
print(meta['name'], '| qubits:', op.num_qubits, '| HF bitstring:', meta['hf_bitstring'])
print('cost:', pauli_cost(op))                       # Pauli terms (no identity) and QWC groups
print('E_HF  =', meta['e_hf_hartree'])
print('E_FCI =', meta['e_fci_full_space_hartree'], '(organizer reference)')
print('exact ground of this operator =', exact_ground_energy(op, const))

## 2. A short VQE run (example ansatz, COBYLA)

Initial angles are drawn from U(−π, π). Starting from all-zero angles with an HF initial state usually stalls at the HF energy: by Brillouin's theorem the energy gradient is zero there.

In [ ]:
ansatz, params = hardware_efficient_ansatz(op.num_qubits, reps=2, initial=hf_circuit(meta['hf_bitstring']))
cost = CountedCost(lambda x: energy_statevector(ansatz.assign_parameters(dict(zip(params, x))), op, const))
x0 = np.random.default_rng(0).uniform(-np.pi, np.pi, len(params))
res = minimize(cost, x0, method='COBYLA', options={'maxiter': 200, 'rhobeg': 1.0})
bound = ansatz.assign_parameters(dict(zip(params, res.x)))
print(f"E_VQE = {res.fun:.6f} Ha | error = {(res.fun - meta['e_fci_full_space_hartree'])*1000:.3f} mHa | optimizer calls = {cost.calls}")

## 3. Shot-based energy with standard error (ideal and noisy on Processor D)

In [ ]:
D = load_processor('processor_D_template.json')
print('Processor D check:', validate_processor_d(D, op.num_qubits) or 'OK')
for shots in SHOT_SETTINGS:
    ideal = energy_shots(bound, op, const, shots_per_group=shots, noisy=False)
    noisy = energy_shots(bound, op, const, shots_per_group=shots, noisy=True, processor=D)
    print(f"{shots:5d} shots/group: ideal {ideal['energy']:.4f} ± {ideal['std_error']:.4f} | "
          f"noisy(D) {noisy['energy']:.4f} ± {noisy['std_error']:.4f} | groups {ideal['groups']}, total shots {ideal['total_shots']}")

## 4. Resource metrics on Processors A, B and D

In [ ]:
for P in (PROCESSOR_A, PROCESSOR_B, D):
    print(json.dumps(circuit_metrics(bound, P)))

## 5. Objective 1 — Bell state on Processor D

In [ ]:
for pair in [(0, 1), (0, 8)]:
    for noisy in (False, True):
        r = bell_experiment(D, pair, noisy)
        print(pair, 'noisy' if noisy else 'ideal', f"XX={r['XX']:+.3f} YY={r['YY']:+.3f} ZZ={r['ZZ']:+.3f} F={r['fidelity']:.4f}±{r['fidelity_se']:.4f}")

## 6. SPSA and the PES helpers (synthetic data, for usage only)

In [ ]:
c2 = CountedCost(lambda x: energy_statevector(ansatz.assign_parameters(dict(zip(params, x))), op, const))
out = spsa_minimize(c2, x0, maxiter=100)
print('SPSA energy', round(out['fun'], 6), 'calls', c2.calls)
R = np.array([0.6, 0.65, 0.7, 0.75, 0.8]); E = 0.5 * (R - 0.7) ** 2 - 1.0     # synthetic parabola
print(harmonic_analysis(R, E, window=(0.6, 0.8), reduced_mass_amu=0.5))
print('NPE (synthetic):', non_parallelity_error([1.0, 1.2, 1.1], [0.9, 1.0, 1.0]))